# Patent Data Preprocessing Pipeline

This notebook implements a comprehensive data preprocessing pipeline for patent classification using Cooperative Patent Classification (CPC) codes. The pipeline transforms raw patent data from the PatentsView API into training-ready datasets for training a large language model (LLM) on Mistral's fine-tuning API.

## Overview

The preprocessing pipeline consists of several key stages:

1. **Data Acquisition**: Fetching patent records from PatentsView API
2. **Data Cleaning**: Removing duplicates and normalizing CPC classifications
3. **Class Selection**: Focusing on the top 48 CPC classes (90% coverage)
4. **Dataset Splitting**: Creating stratified train/validation/test splits
5. **Text Processing**: Merging with detailed patent descriptions
6. **Text Truncation**: Optimizing text length for model input constraints
7. **Format Conversion**: Preparing data for fine-tuning APIs

In [ ]:
import aiofiles
import httpx
import os
import json
from tqdm.notebook import tqdm
import pandas as pd
import numpy as np
import duckdb
import re
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from mistral_common.protocol.instruct.request import ChatCompletionRequest
from mistral_common.tokens.tokenizers.mistral import MistralTokenizer

pd.set_option("display.max_colwidth", 1500)
pd.set_option("display.max_rows", None)
tqdm.pandas()

train_ds_id = 1  # For logging dataset version

## 1. Data Acquisition

We’ll retrieve 85,000 records of 2024 granted patents and their CPC classes from the PatentsView API and store them in a JSONL file. By capturing this many records, we maximize coverage across the 137 possible CPC classes, while ensuring the data is current and reflects the latest classification guidelines.

**Data retrieved per patent:**
- Patent ID
- Patent grant year
- Patent type
- Patent abstract
- Complete CPC classification hierarchy

An API key for PatentsView is required to access the data and can be obtained from https://patentsview-support.atlassian.net/servicedesk/customer/portal/1/group/1/create/18 (Takes about 1-2 business days to be approved).

In [ ]:
async def fetch_and_write():
    last_patent_id = None

    # Delete patents.jsonl if it exists and create a new one
    if os.path.exists("data/patents.jsonl"):
        os.remove("data/patents.jsonl")

    if os.getenv("PATENT_API_KEY") is None:
        raise ValueError("PATENT_API_KEY is not set")

    async with httpx.AsyncClient() as client:
        for i in tqdm(range(85)):
            # Build URL
            if i == 0:
                o_param = '{"size":1000}'
            else:
                o_param = f'{{"size":1000, "after":{last_patent_id}}}'
            url = f'https://search.patentsview.org/api/v1/patent/?q={{"_and":[{{"_gte":{{"patent_date":"2024-01-01"}}}},{{"_lte":{{"patent_date":"2024-12-31"}}}}]}}&o={o_param}&f=["patent_id", "patent_year", "patent_type", "cpc_current", "patent_abstract"]'

            # Set headers
            headers = {"X-Api-Key": os.getenv("PATENT_API_KEY")}

            # Fetch patents data
            response = await client.get(url, headers=headers, timeout=60)

            # Check if patents data is valid
            try:
                patents = response.json()["patents"]
            except KeyError:
                print(f"Error: patents data is invalid for page {i}")
                continue
            last_patent_id = patents[-1]["patent_id"]

            # Write patents to patents.jsonl
            async with aiofiles.open("data/patents.jsonl", "a") as f:
                for patent in patents:
                    await f.write(json.dumps(patent) + "\n")


await fetch_and_write()

## 2. Data Cleaning and Normalization

In [ ]:
# Load patents.jsonl as a df
df = pd.read_json("data/patents.jsonl", lines=True)

# Deduplicate patent ids
df.drop_duplicates(subset=["patent_id"], inplace=True)

df.head(3)

### CPC Classification Simplification

The CPC system has a hierarchical structure (Section → Class → Subclass → Group), and each patent may be assigned to multiple CPC classes and subclasses. For this project, we focus on class-level predictions to strike the right balance between granularity and efficiency, while keeping time and budget in mind.


In [ ]:
# Unnest cpc_current column
df_exploded = df.explode("cpc_current")
cpc_expanded = df_exploded["cpc_current"].apply(pd.Series)

# Drop unnecessary columns
cpc_expanded = cpc_expanded.drop(
    columns=["cpc_class", "cpc_subclass", "cpc_group", "cpc_group_id", "cpc_sequence"]
)

# Concatenate with the original DataFrame (excluding the original cpc_current column)
df_exploded = pd.concat(
    [df_exploded.drop(columns=["cpc_current"]), cpc_expanded], axis=1
)

# Deduplicate rows
df_exploded = df_exploded.drop_duplicates()

# Consolidate all class_ids and subclass_ids into a single list per patent id
df_clean = df_exploded.groupby(
    ["patent_id", "patent_type", "patent_year", "patent_abstract"], as_index=False
).agg(cpc_class_ids=("cpc_class_id", set), cpc_subclass_ids=("cpc_subclass_id", set))

# Write df_clean to patents_clean.jsonl
df_clean.to_json("data/patents_clean.jsonl", orient="records", lines=True)

## 3. Class Selection Strategy

In analyzing 85,000 patents, we observed a long-tail distribution: 48 CPC classes cover 90% of patents, while 81 classes are rare. For this project, we decided to focus on the top 48 classes as this improves: 
- Training efficiency: More examples per class enable faster, more stable learning.
- Accuracy: Sufficient data reduces overfitting and improves generalization.
- Cost-effectiveness: Fewer rare classes means lower compute and annotation costs, while still covering most real-world needs.

Covering rare classes would require far more data and time — beyond the scope of this small project.

### Retain only top classes

In [ ]:
# Read patents_clean.jsonl as df_clean
df_clean = pd.read_json("data/patents_clean.jsonl", lines=True)

value_counts = df_clean["cpc_class_ids"].explode().value_counts(normalize=True)
print(f"Number of classes in dataset: {len(value_counts)}")
# print(value_counts)

vc_cumsum = value_counts.cumsum()
top_90 = vc_cumsum[vc_cumsum <= 0.9]
# print(top_90)

# Get top classes
top_class_ids = list(top_90.index)
print(f"Number of top classes: {len(top_class_ids)}")
print(top_class_ids)

# Filter df_clean to only include rows where cpc_class_id is in top_class_ids
df_clean_top_classes = df_clean[
    df_clean["cpc_class_ids"].apply(lambda x: any(c in top_class_ids for c in x))
]

# Remove cpc_class_ids that are not in top_class_ids
df_clean_top_classes.loc[:, "cpc_class_ids"] = df_clean_top_classes[
    "cpc_class_ids"
].apply(lambda x: [c for c in x if c in top_class_ids])

# Remove cpc_subclass_ids that are not in top_class_ids
df_clean_top_classes.loc[:, "cpc_subclass_ids"] = df_clean_top_classes[
    "cpc_subclass_ids"
].apply(lambda x: [c for c in x if any(s in c for s in top_class_ids)])

df_clean_top_classes.head(2)

## 4. Dataset Splitting Strategy

We implement a stratified splitting approach to prevent data leakage while maintaining representative class distributions across all splits.

**Three-Stage Splitting Process**
1. **Stage 1**: Split into train+validation (99.7%) vs test (0.3%)
2. **Stage 2**: Split train+validation into train (majority) vs validation (0.3%)  
3. **Stage 3**: Downsample training set for cost-efficient experiments (fraction can be varied for experimentation)

In [ ]:
# Set approx number of train samples
train_frac = 0.025  # 0.013 is ~1k samples

# Create a binary indicator matrix for classes
Y = np.array(
    [
        [int(c in x) for c in top_class_ids]
        for x in df_clean_top_classes["cpc_class_ids"]
    ]
)
# print("Original size:", len(df_clean_top_classes))

# First split: train+val vs test
msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.003, random_state=42)
trainval_idx, test_idx = next(msss1.split(np.zeros(len(df_clean_top_classes)), Y))
df_trainval = df_clean_top_classes.iloc[trainval_idx].copy()
df_test = df_clean_top_classes.iloc[test_idx].copy()

# Second split: train vs val (from trainval)
msss2 = MultilabelStratifiedShuffleSplit(
    n_splits=1, test_size=0.003 / (1 - 0.003), random_state=42
)
train_idx, val_idx = next(msss2.split(np.zeros(len(df_trainval)), Y[trainval_idx]))
df_train = df_trainval.iloc[train_idx].copy()
df_val = df_trainval.iloc[val_idx].copy()

# Third split: downsample train
msss3 = MultilabelStratifiedShuffleSplit(
    n_splits=1, test_size=train_frac, random_state=42
)
_, train_idx = next(msss3.split(np.zeros(len(df_train)), Y[train_idx]))
df_train = df_train.iloc[train_idx].copy()

print(f"Train size: {len(df_train)}")
print(f"Val size: {len(df_val)}")
print(f"Test size: {len(df_test)}")
print(f"Total size: {len(df_train) + len(df_val) + len(df_test)}")

In [ ]:
# Check that all train, val, test sets contain all top classes
train_classes = set(df_train["cpc_class_ids"].explode())
test_classes = set(df_test["cpc_class_ids"].explode())
val_classes = set(df_val["cpc_class_ids"].explode())
assert len(train_classes & set(top_class_ids)) == len(top_class_ids)
assert len(test_classes & set(top_class_ids)) == len(top_class_ids)
assert len(val_classes & set(top_class_ids)) == len(top_class_ids)

In [ ]:
# Check for overlap of patent ids in train, test, val
assert df_train.patent_id.isin(df_test.patent_id).sum() == 0
assert df_train.patent_id.isin(df_val.patent_id).sum() == 0
assert df_test.patent_id.isin(df_val.patent_id).sum() == 0

# Check for duplicates in train, test, val
assert df_train.patent_id.duplicated().sum() == 0
assert df_test.patent_id.duplicated().sum() == 0
assert df_val.patent_id.duplicated().sum() == 0

In [ ]:
# Save df_train, df_test, df_val to jsonl
df_train.to_json(f"data/df_ds{train_ds_id}_train.jsonl", orient="records", lines=True)
df_test.to_json("data/df_test.jsonl", orient="records", lines=True)
df_val.to_json("data/df_val.jsonl", orient="records", lines=True)

## 5. Patent Description Integration

Patent abstracts from the API provide only limited information, while full patent descriptions contain the detailed technical content and context needed for accurate classification. Using full descriptions improves training quality, enables models to learn nuanced patterns, and better reflects how patent examiners classify in practice.

The detailed patent description texts are available from patentsview.org/download. Since the full file is around 20GB, we will extract texts only for the selected patent IDs to avoid processing the entire dataset, and then merge them with the corresponding CPC codes.

In [ ]:
# Read train, test, val data from data folder
df_train = pd.read_json(f"data/df_ds{train_ds_id}_train.jsonl", lines=True)
df_test = pd.read_json("data/df_test.jsonl", lines=True)
df_val = pd.read_json("data/df_val.jsonl", lines=True)

# Get patent ids from train, test, val data
train_patent_ids = set(df_train["patent_id"].tolist())
test_patent_ids = set(df_test["patent_id"].tolist())
val_patent_ids = set(df_val["patent_id"].tolist())
all_patent_ids = list(train_patent_ids.union(test_patent_ids).union(val_patent_ids))

In [ ]:
# Clean malformed rows in tsv file
# Run this cell only if g_detail_desc_text_2024_fixed.tsv does not exist
if os.path.exists("g_detail_desc_text_2024_fixed.tsv"):
    raise SystemExit("g_detail_desc_text_2024_fixed.tsv already exists")

input_path = "g_detail_desc_text_2024.tsv"
output_path = "g_detail_desc_text_2024_fixed.tsv"

patent_id_pattern = re.compile(r'^"\d+"')
with open(input_path, "r", encoding="utf-8") as infile:
    header = infile.readline()
    total_records = sum(1 for line in infile if patent_id_pattern.match(line))

num_records_to_process = max(
    1, int(total_records * 1)
)  # 1 means 100% of records will be processed

with (
    open(input_path, "r", encoding="utf-8") as infile,
    open(output_path, "w", encoding="utf-8") as outfile,
):
    header = infile.readline()
    outfile.write(header)
    current_row = ""
    records_written = 0

    with tqdm(total=num_records_to_process, desc="Writing records") as pbar:
        for line in infile:
            # If we have a valid patent ID, write the current row
            if patent_id_pattern.match(line):
                if current_row:
                    outfile.write(current_row)
                    records_written += 1
                    pbar.update(1)
                    if records_written >= num_records_to_process:
                        break
                current_row = line
            # If we don't have a valid patent ID, append the line to the current row
            else:
                current_row = current_row.rstrip("\n") + " " + line.lstrip("\n")
        if records_written < num_records_to_process and current_row:
            outfile.write(current_row)
            pbar.update(1)

In [ ]:
# Load patent descriptions from g_detail_desc_text_2024_fixed.tsv
con = duckdb.connect(database=":memory:")
data = con.read_csv(
    "g_detail_desc_text_2024_fixed.tsv",
    delimiter="\t",
    quotechar='"',
    escapechar='"',
    ignore_errors=True,
    null_padding=True,
    strict_mode=False,
)

# Filter patent_ids that are in all_patent_ids
con.execute(
    "CREATE TABLE temp_ids AS SELECT * FROM (SELECT UNNEST(?) AS patent_id)",
    [all_patent_ids],
)
filtered_data = con.execute("""
    SELECT * FROM data
    WHERE patent_id IN (
        SELECT CAST(patent_id AS INTEGER) FROM temp_ids
    )
""").df()

len(filtered_data)

In [ ]:
# Merge df_train/df_test/df_val and filtered_data on patent_id col
df_train_merged = pd.merge(df_train, filtered_data, on="patent_id", how="inner")
df_test_merged = pd.merge(df_test, filtered_data, on="patent_id", how="inner")
df_val_merged = pd.merge(df_val, filtered_data, on="patent_id", how="inner")

# Save merged dataframes to jsonl files
df_train_merged.to_json(
    f"data/df_ds{train_ds_id}_train_merged.jsonl", orient="records", lines=True
)
df_test_merged.to_json("data/df_test_merged.jsonl", orient="records", lines=True)
df_val_merged.to_json("data/df_val_merged.jsonl", orient="records", lines=True)

## Patent Text Preprocessing and Optimization

While modern models have long context windows, raw patent descriptions can exceed 20,000 words. We apply a `process_description()` pipeline to truncate texts to 5,000 words, ensuring token and cost efficiency while retaining key technical content.

In [ ]:
# Helper function to count tokens
def count_tokens(prompt):
    tokenizer = MistralTokenizer.v3(is_tekken=True)
    model_name = "ministral-8b-2410"
    tokenizer = MistralTokenizer.from_model(model_name)

    # Tokenize a list of messages
    tokenized = tokenizer.encode_chat_completion(
        ChatCompletionRequest(
            messages=[
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            model=model_name,
        )
    )
    tokens = tokenized.tokens
    return len(tokens)


# Helper function to remove tables from patent description
def remove_tables(text):
    # Matches 'TABLE' (all caps), followed by any characters (non-greedy), up to a double newline or end of string.
    pattern = re.compile(r"TABLE[\s\S]+?(?=(\n\n|$))")
    return re.sub(pattern, "", text)


# Helper function to process description
def process_description(desc):
    words = desc.split(" ")
    if len(words) < 5000:
        return desc

    # Truncate to 5000 words and add ellipsis
    truncated_words = words[:5000]
    desc = " ".join(truncated_words) + " ..."

    # Check token count
    token_count = count_tokens(desc)
    if token_count <= 10000:
        return desc

    # Remove tables if still too long
    desc_no_tables = remove_tables(desc)
    if count_tokens(desc_no_tables) <= 10000:
        return desc_no_tables

    # Remove words longer than 50 characters as a last resort
    filtered_words = [word for word in truncated_words if len(word) <= 50]
    desc = " ".join(filtered_words)
    return desc

In [ ]:
# Process patent desc and truncate if necessary
def process_patent_desc(input_path: str, output_path: str, desc_key: str, process_func):
    # Count total lines in input for progress bar
    with open(input_path, "r", encoding="utf-8") as f:
        total_lines = sum(1 for _ in f)

    # Count lines already processed in output (if resuming)
    processed_lines = 0
    if os.path.exists(output_path):
        with open(output_path, "r", encoding="utf-8") as f:
            processed_lines = sum(1 for _ in f)

    with (
        open(input_path, "r", encoding="utf-8") as infile,
        open(output_path, "a", encoding="utf-8") as outfile,
    ):
        # Skip lines that have already been processed
        for _ in range(processed_lines):
            next(infile)
        # Process remaining lines
        for line in tqdm(
            infile,
            total=total_lines - processed_lines,
            initial=processed_lines,
            desc=f"Processing {input_path}",
        ):
            item = json.loads(line)
            item["patent_desc_trunc"] = process_func(item[desc_key])
            outfile.write(json.dumps(item, ensure_ascii=False) + "\n")


process_patent_desc(
    input_path="data/df_test_merged.jsonl",
    output_path="data/df_test_final.jsonl",
    desc_key="description_text",
    process_func=process_description,
)
process_patent_desc(
    input_path="data/df_val_merged.jsonl",
    output_path="data/df_val_final.jsonl",
    desc_key="description_text",
    process_func=process_description,
)
process_patent_desc(
    input_path=f"data/df_ds{train_ds_id}_train_merged.jsonl",
    output_path=f"data/df_ds{train_ds_id}_train_final.jsonl",
    desc_key="description_text",
    process_func=process_description,
)

## 7. Fine-tuning Format Conversion

The final step transforms our cleaned datasets into the specific format required by Mistral's fine-tuning API. 
```json
{
  "text": "Patent description text...",
  "labels": {"cpc_class_ids": ["G06", "H01", "B60"]}
}
```

In [ ]:
# Prepare files in the format required for fine-tuning
def create_finetuning_file(df_file_name, output_file_name):
    df = pd.read_json(df_file_name, lines=True)

    df["text"] = df["patent_desc_trunc"]
    df["labels"] = df["cpc_class_ids"].progress_apply(lambda x: {"cpc_class_ids": x})

    # Drop all other columns
    df = df[["text", "labels"]]

    df.to_json(output_file_name, orient="records", lines=True, force_ascii=False)
    print(f"Created finetuning file: {output_file_name}")

In [ ]:
train_file_path = f"data/ft_ds{train_ds_id}_train.jsonl"
val_file_path = "data/ft_val.jsonl"
test_file_path = "data/ft_test.jsonl"
create_finetuning_file(f"data/df_ds{train_ds_id}_train_final.jsonl", train_file_path)
create_finetuning_file("data/df_val_final.jsonl", val_file_path)
create_finetuning_file("data/df_test_final.jsonl", test_file_path)